# LeetCode 3691 — Maximum Total Subarray Value II bằng Python

- Link: https://leetcode.com/problems/maximum-total-subarray-value-ii/
- Độ khó: **Hard**
- Chủ đề: **Top-k** bằng heap trên các dãy đơn điệu, **Sparse Table** (RMQ `O(1)`), Segment Tree, tính đơn điệu của `max − min`

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maxTotalValue(self, nums: List[int], k: int) -> int:
```

## 1. Mục tiêu học

1. Thấy rằng chọn `k` mảng con **phân biệt** để tổng giá trị lớn nhất ⇔ lấy **`k` giá trị lớn nhất** trong `n(n+1)/2` giá trị.
2. Khai thác tính đơn điệu: với `l` cố định, `value(l, r) = max − min` **không giảm** theo `r`.
3. Bài "top-k trên `n` dãy đã sắp" ⇒ **heap** chứa phần tử đầu của mỗi dãy, lấy ra rồi đẩy phần tử kế tiếp.
4. Truy vấn `max`/`min` đoạn trong `O(1)` bằng **sparse table**.
5. So sánh với cách "ma trận Young" (mở rộng từ `(0, n−1)` sang hai láng giềng) và với vét cạn.

## 2. Đề bài & ví dụ

Chọn **đúng `k`** mảng con phân biệt (khác `(l, r)`), có thể chồng nhau. Giá trị mảng con = `max − min`. Tối đa hoá tổng giá trị.

**Ví dụ 1:** `nums = [1,3,2], k = 2` → `4`.

**Ví dụ 2:** `nums = [4,2,5,1], k = 3` → `12`.

**Ràng buộc:** `n ≤ 5·10^4`, `0 ≤ nums[i] ≤ 10^9`, `k ≤ min(10^5, n(n+1)/2)`.

In [1]:
from typing import List
import heapq, random, time

examples = [([1, 3, 2], 2, 4), ([4, 2, 5, 1], 3, 12)]
for a, k, exp in examples:
    print(a, k, "-> kỳ vọng", exp)

[1, 3, 2] 2 -> kỳ vọng 4
[4, 2, 5, 1] 3 -> kỳ vọng 12


## 3. Kiến thức nền

### 3.1. Đơn điệu theo bao hàm

Nếu `[l, r] ⊆ [l', r']` thì `max` không giảm, `min` không tăng ⇒ `value(l, r) ≤ value(l', r')`. Đặc biệt, với `l` cố định, `value(l, r)` không giảm theo `r`; với `r` cố định, không tăng theo `l`.

In [2]:
a = [4, 2, 5, 1]; n = len(a)
for l in range(n):
    print(f"l={l}:", [max(a[l:r + 1]) - min(a[l:r + 1]) for r in range(l, n)])

l=0: [0, 2, 3, 4]
l=1: [0, 3, 4]
l=2: [0, 4]
l=3: [0]


### 3.2. Sparse Table

`mx[j][i] = max(a[i .. i + 2^j − 1])`. Truy vấn `[l, r]`: `t = ⌊log₂(r − l + 1)⌋`, kết quả `max(mx[t][l], mx[t][r − 2^t + 1])` — hai khối chồng nhau phủ kín đoạn. Dựng `O(n log n)`, truy vấn `O(1)`.

In [3]:
def build_sparse(a, op):
    tab = [a[:]]; j = 1
    while (1 << j) <= len(a):
        prev = tab[-1]; h = 1 << (j - 1)
        tab.append([op(prev[i], prev[i + h]) for i in range(len(a) - (1 << j) + 1)]); j += 1
    return tab

def query(tab, l, r, op):
    t = (r - l + 1).bit_length() - 1
    return op(tab[t][l], tab[t][r - (1 << t) + 1])

mxT = build_sparse([4, 2, 5, 1], max)
print("max[1..3] =", query(mxT, 1, 3, max))

max[1..3] = 5


### 3.3. Top-k trên nhiều dãy đã sắp

Có `n` dãy, mỗi dãy giảm dần. Đưa phần tử đầu (lớn nhất) của mỗi dãy vào max-heap; mỗi lần lấy ra phần tử lớn nhất, đẩy phần tử kế tiếp của cùng dãy. Sau `k` lần lấy, ta có đúng `k` phần tử lớn nhất — `O((n + k) log n)`.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Vì không có ràng buộc gì giữa các mảng con được chọn (ngoài phân biệt), đáp án là tổng `k` giá trị lớn nhất trong multiset `{value(l, r)}`.

Với mỗi `l`, dãy `value(l, n−1), value(l, n−2), …, value(l, l)` **giảm dần** (không tăng). Vậy có `n` dãy đã sắp ⇒ áp dụng 3.3 với phần tử đầu của dãy `l` là `value(l, n−1)`.

Ví dụ 2: `value(0,3) = value(1,3) = value(2,3) = 4` ⇒ ba lần lấy đầu tiên đều 4 ⇒ `12`. ✔

## 5. Cách 1 — Brute Force: liệt kê mọi giá trị

In [4]:
def brute(nums, k):
    n = len(nums); vals = []
    for l in range(n):
        mx = mn = nums[l]
        for r in range(l, n):
            mx = max(mx, nums[r]); mn = min(mn, nums[r]); vals.append(mx - mn)
    vals.sort(reverse=True)
    return sum(vals[:k])

for a, k, exp in examples:
    print(brute(a, k), "kỳ vọng", exp)

4 kỳ vọng 4
12 kỳ vọng 12


**Nút thắt:** `n(n+1)/2 ≈ 1,25·10^9` giá trị với `n = 5·10^4`.

## 6. Ý tưởng thuật toán / bước đột phá

1. Dựng sparse table cho `max` và `min`.
2. Heap ban đầu: `(−value(l, n−1), l, n−1)` cho mọi `l`.
3. Lặp `k` lần: lấy `(−v, l, r)`, cộng `v`; nếu `r > l`, đẩy `(−value(l, r−1), l, r−1)`.

Mỗi phần tử heap ứng với một mảng con khác nhau ⇒ tự nhiên **phân biệt**. Tổng `O(n log n + k log n)`.

## 7. Các cách tiếp theo

- **Cách 2:** heap + **segment tree** cho `max/min` đoạn — `O((n + k) log n · log n)`.
- **Cách 3:** heap + **sparse table** — `O(n log n + k log n)` (lời giải nộp).
- **Cách 4:** "ma trận Young": bắt đầu từ `(0, n−1)` (giá trị lớn nhất), mỗi lần lấy `(l, r)` đẩy hai láng giềng `(l+1, r)` và `(l, r−1)` (có tập `visited`) — `O(k log k)` truy vấn.

### 7.1. Cách 2 — Heap + segment tree

In [5]:
class SegTree:
    def __init__(self, a):
        self.n = 1
        while self.n < len(a): self.n *= 2
        self.mx = [-1] * (2 * self.n); self.mn = [float("inf")] * (2 * self.n)
        for i, v in enumerate(a): self.mx[self.n + i] = v; self.mn[self.n + i] = v
        for i in range(self.n - 1, 0, -1):
            self.mx[i] = max(self.mx[2 * i], self.mx[2 * i + 1]); self.mn[i] = min(self.mn[2 * i], self.mn[2 * i + 1])
    def value(self, l, r):
        hi, lo = -1, float("inf"); l += self.n; r += self.n + 1
        while l < r:
            if l & 1: hi = max(hi, self.mx[l]); lo = min(lo, self.mn[l]); l += 1
            if r & 1: r -= 1; hi = max(hi, self.mx[r]); lo = min(lo, self.mn[r])
            l //= 2; r //= 2
        return hi - lo

def heap_segtree(nums, k):
    n = len(nums); st = SegTree(nums)
    h = [(-st.value(l, n - 1), l, n - 1) for l in range(n)]; heapq.heapify(h); total = 0
    for _ in range(k):
        v, l, r = heapq.heappop(h); total -= v
        if r > l: heapq.heappush(h, (-st.value(l, r - 1), l, r - 1))
    return total

for a, k, exp in examples:
    print(heap_segtree(a, k), "kỳ vọng", exp)

4 kỳ vọng 4
12 kỳ vọng 12


### 7.2. Cách 3 — Heap + sparse table (lời giải chính)

In [6]:
def heap_sparse(nums, k):
    n = len(nums)
    MX = build_sparse(nums, max); MN = build_sparse(nums, min)
    def value(l, r):
        t = (r - l + 1).bit_length() - 1; s = r - (1 << t) + 1
        return max(MX[t][l], MX[t][s]) - min(MN[t][l], MN[t][s])
    h = [(-value(l, n - 1), l, n - 1) for l in range(n)]; heapq.heapify(h); total = 0
    for _ in range(k):
        v, l, r = heapq.heappop(h); total -= v
        if r > l: heapq.heappush(h, (-value(l, r - 1), l, r - 1))
    return total

for a, k, exp in examples:
    print(heap_sparse(a, k), "kỳ vọng", exp)

4 kỳ vọng 4
12 kỳ vọng 12


### 7.3. Cách 4 — Mở rộng kiểu "ma trận Young"

Xem `value(l, r)` (với `l ≤ r`) như một bảng tam giác: giảm (không tăng) khi đi xuống (`l` tăng) hoặc sang trái (`r` giảm). Phần tử lớn nhất ở góc `(0, n−1)`. Mỗi phần tử lấy ra mở hai ô láng giềng; ô nào cũng chỉ được đẩy vào heap một lần.

In [7]:
def young(nums, k):
    n = len(nums)
    MX = build_sparse(nums, max); MN = build_sparse(nums, min)
    def value(l, r):
        t = (r - l + 1).bit_length() - 1; s = r - (1 << t) + 1
        return max(MX[t][l], MX[t][s]) - min(MN[t][l], MN[t][s])
    h = [(-value(0, n - 1), 0, n - 1)]; seen = {(0, n - 1)}; total = 0
    for _ in range(k):
        v, l, r = heapq.heappop(h); total -= v
        for nl, nr in ((l + 1, r), (l, r - 1)):
            if nl <= nr and (nl, nr) not in seen:
                seen.add((nl, nr)); heapq.heappush(h, (-value(nl, nr), nl, nr))
    return total

for a, k, exp in examples:
    print(young(a, k), "kỳ vọng", exp)

4 kỳ vọng 4
12 kỳ vọng 12


**Vì sao Cách 4 đúng?** Mọi ô `(l, r)` khác góc đều có ít nhất một "cha" `(l−1, r)` hoặc `(l, r+1)` với giá trị `≥`; khi cha được lấy ra, ô được đẩy vào. Theo quy nạp, ô thứ `k` lớn nhất luôn có mặt trong heap khi đến lượt.

## 8. TRACE từng bước

In [8]:
def trace(nums, k):
    n = len(nums)
    MX = build_sparse(nums, max); MN = build_sparse(nums, min)
    value = lambda l, r: query(MX, l, r, max) - query(MN, l, r, min)
    h = [(-value(l, n - 1), l, n - 1) for l in range(n)]; heapq.heapify(h); total = 0
    print("heap ban đầu:", sorted((-v, l, r) for v, l, r in h), "(giá trị, l, r)")
    for step in range(1, k + 1):
        v, l, r = heapq.heappop(h); total -= v
        msg = f"lấy {nums[l:r + 1]} (l={l}, r={r}) giá trị {-v} -> tổng {total}"
        if r > l:
            nv = value(l, r - 1); heapq.heappush(h, (-nv, l, r - 1)); msg += f"; đẩy (l={l}, r={r - 1}) giá trị {nv}"
        print(f"  bước {step}: {msg}")
    return total

trace([4, 2, 5, 1], 5)
print("brute:", brute([4, 2, 5, 1], 5))

heap ban đầu: [(0, 3, 3), (4, 0, 3), (4, 1, 3), (4, 2, 3)] (giá trị, l, r)
  bước 1: lấy [4, 2, 5, 1] (l=0, r=3) giá trị 4 -> tổng 4; đẩy (l=0, r=2) giá trị 3
  bước 2: lấy [2, 5, 1] (l=1, r=3) giá trị 4 -> tổng 8; đẩy (l=1, r=2) giá trị 3
  bước 3: lấy [5, 1] (l=2, r=3) giá trị 4 -> tổng 12; đẩy (l=2, r=2) giá trị 0
  bước 4: lấy [4, 2, 5] (l=0, r=2) giá trị 3 -> tổng 15; đẩy (l=0, r=1) giá trị 2
  bước 5: lấy [2, 5] (l=1, r=2) giá trị 3 -> tổng 18; đẩy (l=1, r=1) giá trị 0
brute: 18


## 9. Code nộp LeetCode

In [9]:
import heapq

class Solution:
    def maxTotalValue(self, nums: List[int], k: int) -> int:
        n = len(nums)
        MX = [nums[:]]
        MN = [nums[:]]
        j = 1
        while (1 << j) <= n:
            h = 1 << (j - 1)
            pm, pn = MX[-1], MN[-1]
            size = n - (1 << j) + 1
            MX.append([pm[i] if pm[i] > pm[i + h] else pm[i + h] for i in range(size)])
            MN.append([pn[i] if pn[i] < pn[i + h] else pn[i + h] for i in range(size)])
            j += 1

        def value(l, r):
            t = (r - l + 1).bit_length() - 1
            s = r - (1 << t) + 1
            a, b = MX[t][l], MX[t][s]
            c, d = MN[t][l], MN[t][s]
            return (a if a > b else b) - (c if c < d else d)

        heap = [(-value(l, n - 1), l, n - 1) for l in range(n)]
        heapq.heapify(heap)
        total = 0
        for _ in range(k):
            v, l, r = heapq.heappop(heap)
            total -= v
            if r > l:
                heapq.heappush(heap, (-value(l, r - 1), l, r - 1))
        return total

sol = Solution()
for a, k, exp in examples:
    got = sol.maxTotalValue(a, k)
    print(a, k, "->", got, "✔" if got == exp else "✘")

[1, 3, 2] 2 -> 4 ✔
[4, 2, 5, 1] 3 -> 12 ✔


## 10. Vì sao thuật toán đúng

1. **Quy về top-k:** các mảng con được chọn độc lập (chỉ cần phân biệt) và tổng là cộng tính ⇒ tối ưu khi chọn `k` mảng con có giá trị lớn nhất (đổi bất kỳ mảng con nhỏ hơn lấy một mảng con lớn hơn chưa chọn không làm giảm tổng).
2. **Dãy đã sắp:** với `l` cố định, `value(l, r)` không giảm theo `r` (3.1) ⇒ dãy `r = n−1, n−2, …, l` không tăng.
3. **Heap:** bất biến — heap chứa, với mỗi `l` chưa dùng hết, đúng một phần tử: ô lớn nhất **chưa lấy** của dãy `l`. Phần tử lớn nhất chưa lấy toàn cục là đầu của một dãy nào đó ⇒ nằm ở đỉnh heap. Mỗi `(l, r)` được lấy tối đa một lần ⇒ các mảng con phân biệt.
4. **Sparse table** trả `max/min` đúng vì hai khối `[l, l+2^t−1]` và `[r−2^t+1, r]` phủ kín `[l, r]` (phép `max/min` lũy đẳng nên chồng lấn không sao).

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(n² log n)` | `O(n²)` |
| 2. Heap + segment tree | `O((n + k) log² n)` | `O(n)` |
| 3. Heap + sparse table | `O(n log n + k log n)` | `O(n log n)` |
| 4. Young + sparse table | `O(n log n + k log k)` | `O(n log n + k)` |

## 12. Bộ test

In [10]:
METHODS = [brute, heap_segtree, heap_sparse, young, lambda a, k: Solution().maxTotalValue(a, k)]
for a, k, exp in examples:
    assert all(f(a, k) == exp for f in METHODS)
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [11]:
# 12.2 ca biên
assert Solution().maxTotalValue([7], 1) == 0
assert Solution().maxTotalValue([5, 5, 5], 6) == 0
assert Solution().maxTotalValue([0, 10**9], 3) == 10**9                  # k = n(n+1)/2: lấy hết
assert Solution().maxTotalValue([3, 1, 2], 6) == brute([3, 1, 2], 6)
print("Ca biên: OK")

Ca biên: OK


In [12]:
# 12.3 ngẫu nhiên
rng = random.Random(3691)
for _ in range(1000):
    n = rng.randint(1, 12); a = [rng.randint(0, 9) for _ in range(n)]; k = rng.randint(1, n * (n + 1) // 2)
    b = brute(a, k)
    assert all(f(a, k) == b for f in METHODS[1:]), (a, k)
print("1000 test ngẫu nhiên: OK")

1000 test ngẫu nhiên: OK


In [13]:
# 12.4 test lớn: n = 5·10^4, k = 10^5
rng = random.Random(1)
a = [rng.randint(0, 10**9) for _ in range(5 * 10**4)]
t0 = time.time(); x = Solution().maxTotalValue(a, 10**5); t1 = time.time()
y = young(a, 10**5); t2 = time.time()
print("đáp án:", x, "| heap theo l:", round(t1 - t0, 2), "s | Young:", round(t2 - t1, 2), "s | khớp:", x == y)

đáp án: 99997985200000 | heap theo l: 0.28 s | Young: 0.3 s | khớp: True


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ chọn các mảng con **không chồng nhau**

Đề cho phép chồng nhau; chỉ cấm chọn trùng `(l, r)`.

In [14]:
def wrong_disjoint(nums, k):
    # SAI: chọn tham lam các mảng con không chồng nhau (ví dụ chia mảng) — bỏ lỡ các mảng con chồng nhau có giá trị lớn
    n = len(nums); best = 0
    from itertools import combinations
    cuts_all = range(1, n)
    for c in range(0, n):
        for cuts in combinations(cuts_all, c):
            if c + 1 < k: continue
            b = [0] + list(cuts) + [n]
            vals = sorted((max(nums[b[i]:b[i + 1]]) - min(nums[b[i]:b[i + 1]]) for i in range(c + 1)), reverse=True)
            best = max(best, sum(vals[:k]))
    return best

print("sai:", wrong_disjoint([4, 2, 5, 1], 3), "| đúng:", brute([4, 2, 5, 1], 3))

sai: 4 | đúng: 12


### Lỗi 2 — Heap chỉ đẩy một phía, bắt đầu từ mảng con ngắn nhất

Nếu mỗi dãy bắt đầu từ `r = l` (giá trị 0) rồi **tăng** `r`, đỉnh heap không còn là giá trị lớn nhất chưa lấy.

In [15]:
def wrong_start_short(nums, k):
    n = len(nums)
    value = lambda l, r: max(nums[l:r + 1]) - min(nums[l:r + 1])
    h = [(-value(l, l), l, l) for l in range(n)]; heapq.heapify(h); t = 0           # SAI: dãy tăng dần
    for _ in range(k):
        v, l, r = heapq.heappop(h); t -= v
        if r < n - 1: heapq.heappush(h, (-value(l, r + 1), l, r + 1))
    return t

print("sai:", wrong_start_short([4, 2, 5, 1], 3), "| đúng:", brute([4, 2, 5, 1], 3))

sai: 5 | đúng: 12


### Lỗi 3 — Cách 4 quên tập `visited` (một ô bị đẩy hai lần ⇒ chọn trùng)

In [16]:
def wrong_no_visited(nums, k):
    n = len(nums)
    value = lambda l, r: max(nums[l:r + 1]) - min(nums[l:r + 1])
    h = [(-value(0, n - 1), 0, n - 1)]; t = 0
    for _ in range(k):
        v, l, r = heapq.heappop(h); t -= v
        for nl, nr in ((l + 1, r), (l, r - 1)):
            if nl <= nr: heapq.heappush(h, (-value(nl, nr), nl, nr))                # SAI
    return t

rng = random.Random(31)
while True:
    a = [rng.randint(0, 9) for _ in range(rng.randint(3, 7))]; k = rng.randint(2, len(a) * (len(a) + 1) // 2)
    if wrong_no_visited(a, k) != brute(a, k): break
print("phản ví dụ:", a, "k =", k, "| sai:", wrong_no_visited(a, k), "| đúng:", brute(a, k))

phản ví dụ: [5, 3, 3, 5, 6, 5] k = 21 | sai: 51 | đúng: 31


### Lỗi 4 — Tính `max/min` bằng slicing mỗi lần

`max(nums[l:r+1])` tốn `O(n)` ⇒ `k · n = 5·10^9`. Phải dùng sparse table hoặc segment tree.

In [17]:
a = list(range(5 * 10**4))
t0 = time.time()
for _ in range(200): max(a[0:]) - min(a[0:])
print("200 lần slicing toàn mảng:", round(time.time() - t0, 3), "s  -> 10^5 lần sẽ mất khoảng", round((time.time() - t0) * 500, 1), "s")

200 lần slicing toàn mảng: 0.391 s  -> 10^5 lần sẽ mất khoảng 195.3 s


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Chọn `k` phần tử phân biệt, tối đa tổng" + không ràng buộc khác ⇒ **top-k**.
- Top-k trên tập lớn có cấu trúc đơn điệu ⇒ heap "đầu của mỗi dãy" hoặc "ma trận Young".
- Truy vấn `max/min` đoạn tĩnh nhiều lần ⇒ sparse table `O(1)`.
- Bản I của bài (`k` nhỏ hoặc chỉ một mảng con) thường gợi ý `value` lớn nhất là `max(nums) − min(nums)`.

## 15. Biến thể của bài

1. **Bản I** (chọn `k` mảng con, **được** chọn trùng): đáp án `k · (max − min)` toàn mảng.
2. **Giá trị = tổng mảng con** (top-k tổng mảng con): không đơn điệu theo bao hàm ⇒ cần cấu trúc khác (heap trên tiền tố + persistent/sparse).
3. **Tối thiểu** tổng `k` giá trị: lấy `k` giá trị nhỏ nhất — dãy theo `l` đi từ `r = l` tăng dần.
4. **Đếm số mảng con có giá trị ≥ T**: hai con trỏ + deque đơn điệu.

In [18]:
# Biến thể 4: đếm số mảng con có max - min >= T bằng hai con trỏ + deque đơn điệu
from collections import deque

def count_at_least(nums, T):
    n = len(nums); dqmax = deque(); dqmin = deque(); l = 0; less = 0
    for r, x in enumerate(nums):                            # đếm mảng con có max-min < T, rồi lấy phần bù
        while dqmax and nums[dqmax[-1]] <= x: dqmax.pop()
        dqmax.append(r)
        while dqmin and nums[dqmin[-1]] >= x: dqmin.pop()
        dqmin.append(r)
        while dqmax and dqmin and nums[dqmax[0]] - nums[dqmin[0]] >= T:     # với T = 0 có thể rút rỗng cửa sổ
            l += 1
            if dqmax[0] < l: dqmax.popleft()
            if dqmin[0] < l: dqmin.popleft()
        less += r - l + 1
    return n * (n + 1) // 2 - less

rng = random.Random(15)
for _ in range(400):
    a = [rng.randint(0, 9) for _ in range(rng.randint(1, 12))]; T = rng.randint(0, 10)
    exp = sum(1 for l in range(len(a)) for r in range(l, len(a)) if max(a[l:r + 1]) - min(a[l:r + 1]) >= T)
    assert count_at_least(a, T) == exp
print("Biến thể 4 OK:", count_at_least([4, 2, 5, 1], 4))

Biến thể 4 OK: 3


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Khi nào dùng |
|---|---|---|---|
| 1. Vét cạn | Mọi giá trị | `O(n²)` | Kiểm chứng |
| 2. Heap + segment tree | Top-k theo `l` | `O((n+k) log² n)` | Khi có cập nhật |
| 3. Heap + sparse table | Top-k theo `l` | `O((n+k) log n)` | **Lời giải chuẩn** |
| 4. Young + sparse table | Mở rộng hai láng giềng | `O(n log n + k log k)` | Khi `k ≪ n` |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `maxTotalValue([1,2,3,4], 1)`.

**Bài 2 (tính tay).** `maxTotalValue([3,1,2], 3)`.

**Bài 3 (điền TODO).** Viết `min_total_value(nums, k)` — tổng **nhỏ nhất** của `k` mảng con phân biệt.

**Bài 4 (tự viết lại).** Viết lời giải dùng tìm nhị phân trên ngưỡng `T` + `count_at_least` để tìm giá trị lớn thứ `k`, rồi tính tổng (gợi ý: cần thêm **tổng** các giá trị `> T` — có thể tính bằng heap tới ngưỡng, hoặc chấp nhận `O(n²)` cho bài tập).

**Bài 5 (mini challenge).** Trả về danh sách `k` cặp `(l, r)` đạt tối ưu.

In [19]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def min_total_value(nums, k):
    # TODO: mỗi dãy bắt đầu từ r = l (giá trị 0), tăng r
    pass

In [20]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def chosen_subarrays(nums, k):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Lấy mảng con lớn nhất. (2) Cả mảng: `4 − 1`. (3) `3`.

**Bài 2.** (1) Giá trị: `[3,1]=2`, `[3,1,2]=2`, `[1,2]=1`, còn lại 0. (2) Ba lớn nhất `2, 2, 1`. (3) `5`.

**Bài 3.** (1) Dãy theo `l` tăng khi `r` tăng. (2) Min-heap từ `(l, l)`. (3) Lấy `k` lần.

**Bài 4.** (1) Tìm `T*` = giá trị lớn thứ `k` (nhị phân với `count_at_least`). (2) Tổng = Σ các giá trị `> T*` + `(k − #{> T*})·T*`. (3) Kiểm chứng với lời giải chính.

**Bài 5.** (1) Ghi lại `(l, r)` mỗi lần lấy. (2) Kiểm tra phân biệt. (3) Tổng giá trị khớp đáp án.

## 19. Đáp án tham khảo cho bài tập

In [21]:
# Bài 1, 2
assert Solution().maxTotalValue([1, 2, 3, 4], 1) == 3
assert Solution().maxTotalValue([3, 1, 2], 3) == 5 == brute([3, 1, 2], 3)
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [22]:
# Bài 3, 4
def min_total_value_sol(nums, k):
    n = len(nums); MX = build_sparse(nums, max); MN = build_sparse(nums, min)
    value = lambda l, r: query(MX, l, r, max) - query(MN, l, r, min)
    h = [(0, l, l) for l in range(n)]; t = 0
    for _ in range(k):
        v, l, r = heapq.heappop(h); t += v
        if r < n - 1: heapq.heappush(h, (value(l, r + 1), l, r + 1))
    return t

def via_threshold(nums, k):
    n = len(nums); lo, hi = 0, max(nums) - min(nums)
    while lo < hi:                                         # T* lớn nhất với count_at_least(T*) >= k
        mid = (lo + hi + 1) // 2
        if count_at_least(nums, mid) >= k: lo = mid
        else: hi = mid - 1
    T = lo
    above = [max(nums[l:r + 1]) - min(nums[l:r + 1]) for l in range(n) for r in range(l, n)]
    big = [v for v in above if v > T]
    return sum(big) + (k - len(big)) * T

rng = random.Random(4)
for _ in range(400):
    n = rng.randint(1, 10); a = [rng.randint(0, 9) for _ in range(n)]; k = rng.randint(1, n * (n + 1) // 2)
    vals = sorted(max(a[l:r + 1]) - min(a[l:r + 1]) for l in range(n) for r in range(l, n))
    assert min_total_value_sol(a, k) == sum(vals[:k])
    assert via_threshold(a, k) == brute(a, k)
print("Bài 3, 4 OK")

Bài 3, 4 OK


In [23]:
# Bài 5
def chosen_subarrays_sol(nums, k):
    n = len(nums); MX = build_sparse(nums, max); MN = build_sparse(nums, min)
    value = lambda l, r: query(MX, l, r, max) - query(MN, l, r, min)
    h = [(-value(l, n - 1), l, n - 1) for l in range(n)]; heapq.heapify(h); out = []
    for _ in range(k):
        v, l, r = heapq.heappop(h); out.append((l, r))
        if r > l: heapq.heappush(h, (-value(l, r - 1), l, r - 1))
    return out

ch = chosen_subarrays_sol([4, 2, 5, 1], 4)
print(ch, "| tổng:", sum(max([4, 2, 5, 1][l:r + 1]) - min([4, 2, 5, 1][l:r + 1]) for l, r in ch))
assert len(set(ch)) == 4 and sum(max([4, 2, 5, 1][l:r + 1]) - min([4, 2, 5, 1][l:r + 1]) for l, r in ch) == brute([4, 2, 5, 1], 4)

[(0, 3), (1, 3), (2, 3), (0, 2)] | tổng: 15


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
MX, MN = sparse table max / min
value(l, r) = max(MX[t][l], MX[t][r-2^t+1]) - min(MN[t][l], MN[t][r-2^t+1])
heap = [(-value(l, n-1), l, n-1) for l in range(n)]
lặp k lần: lấy (v, l, r); total += -v; nếu r > l: đẩy (-value(l, r-1), l, r-1)
```

**Checklist:**

- [ ] Mảng con được phép **chồng nhau**, chỉ cấm trùng `(l, r)`?
- [ ] Mỗi dãy bắt đầu từ `r = n − 1` (giá trị lớn nhất của dãy)?
- [ ] `max/min` đoạn bằng sparse table (không slicing)?
- [ ] Cách Young có tập `visited`?
- [ ] Đã so với vét cạn?